# SAR + Optical Fusion — Kaggle Training
**Roham Rasouli Kerahroudi — Sakarya Universitesi Bitirme Projesi**

Her run'da TARGET_EPOCHS'u 8 artir: 8 → 16 → 24 → ... → 80

Dataset olarak `wchao0601/m4-sar` ekli olmali.

In [ ]:
import os
import subprocess
from pathlib import Path

# 1 — Repo kur
os.chdir('/kaggle/working')
if not Path('sar-optical-fusion').exists():
    subprocess.run(['git', 'clone', 'https://github.com/RohamRasouli/sar-optical-fusion.git'], check=True)
os.chdir('sar-optical-fusion')
subprocess.run(['pip', 'install', 'rasterio', '-q'], check=True)
subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)

# 2 — Data root bul
result = subprocess.run(['find', '/kaggle/input', '-type', 'd', '-name', 'train'], capture_output=True, text=True)
train_dirs = [d for d in result.stdout.strip().split('\n') if d]
parents = sorted(set(str(Path(d).parent.parent) for d in train_dirs))
DATA_ROOT = parents[0] if parents else '/kaggle/input/m4-sar/M4-SAR/M4-SAR'
print(f'DATA_ROOT: {DATA_ROOT}')

# 3 — Resume kontrol
RUNS = '/kaggle/working/runs'
os.makedirs(RUNS, exist_ok=True)
best = f'{RUNS}/best.pt'
resume = f'--resume {best}' if Path(best).exists() else ''
print('[RESUME]' if resume else '[FRESH]')

# 4 — Egitim
# Her run'da TARGET_EPOCHS'u 8 artir: 8 -> 16 -> 24 -> 32 ...
TARGET_EPOCHS = 8

cmd = ' '.join([
    'python -m src.train',
    '--config configs/kaggle_p100.yaml',
    f'--data_root "{DATA_ROOT}"',
    f'--output "{RUNS}"',
    f'--epochs {TARGET_EPOCHS}',
    resume
])
print(cmd)
os.system(cmd)
